In [0]:
%sql
SHOW CATALOGS;


In [0]:
try:    cqc_key = dbutils.secrets.get(scope="domiciliarycare", key="api_key")
except Exception as e:
    cqc_key = None
    print(f"Secret not found: {e}")


HEADERS = {
    "Ocp-Apim-Subscription-Key": cqc_key,
    "User-Agent": "HomeSafeKentPipeline/1.0",  # no hyphens
    "Accept": "application/json",
}

In [0]:
dbutils.secrets.listScopes()

In [0]:

import time
import json
import requests
import pandas as pd

BASE_URL = "https://api.service.cqc.org.uk/public/v1"

CQC_SUBSCRIPTION_KEY = dbutils.secrets.get(
    catalog="domiciliarycare", schema="security", key="api_key"
)

HEADERS = {
    "Ocp-Apim-Subscription-Key": CQC_SUBSCRIPTION_KEY,
    "User-Agent": "HomeSafeKentPipeline/1.0",  # no hyphens -- CQC's own flagged quirk
    "Accept": "application/json",
}

print("Key loaded from Unity Catalog secret. Ready for the sanity check below.")

In [0]:
def cqc_get(path, params=None, max_retries=3):
    """GET against the CQC Syndication API with basic 429 backoff and readable errors."""
    url = f"{BASE_URL}{path}"
    resp = None
    for attempt in range(1, max_retries + 1):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if resp.status_code == 200:
            print(200)
            return resp.json()

        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limited (429). Waiting {wait}s before retry {attempt}/{max_retries}...")
            time.sleep(wait)
            continue

        if resp.status_code == 403:
            raise RuntimeError(
                "403 Forbidden -- confirmed by CQC Developer Support: this means the "
                "User-Agent header is missing (or, rarely, contains a hyphen). NOT an "
                "auth/key problem -- check HEADERS['User-Agent'] first."
            )

        if resp.status_code == 502:
            raise RuntimeError(
                "502 Bad Gateway. One live test found this for a missing/wrong key, but "
                "CQC support says missing/invalid key should return 401. If you see 502 "
                "with a real key, treat this as unresolved -- test the 4 auth cases "
                "(no header / empty header / garbage key / missing User-Agent) to isolate it."
            )

        if resp.status_code == 401:
            raise RuntimeError(
                "401 Unauthorized -- key sent but rejected, or missing. Check it's the "
                "Syndication product key specifically (CQC issues keys per product)."
            )

        resp.raise_for_status()

    raise RuntimeError(f"Gave up after {max_retries} attempts (last status {resp.status_code if resp else 'n/a'}).")

In [0]:
sample = cqc_get("/locations", params={"page": 1, "perPage": 1})
sample

In [0]:
# Quick test: just Ashford, small perPage, see the filtered count
test = cqc_get("/locations", params={
    "localAuthority": "Ashford",
    "regulatedActivity": "Personal care",
    "perPage": 5,
    "page": 1
})
print(f"Total matching Ashford + Personal care: {test['total']}")
test["locations"]

In [0]:
KENT_LOCAL_AUTHORITIES = [
    "Ashford", "Canterbury", "Dartford", "Dover", "Folkestone and Hythe",
    "Gravesham", "Maidstone", "Sevenoaks", "Swale", "Thanet",
    "Tonbridge and Malling", "Tunbridge Wells",
]
print(f"{len(KENT_LOCAL_AUTHORITIES)} Kent districts configured.")

In [0]:
def fetch_all_locations(local_authorities, regulated_activity="Personal care", per_page=1000, polite_delay=0.3):
    base_params = [("localAuthority", la) for la in local_authorities]
    base_params.append(("regulatedActivity", regulated_activity))
    base_params.append(("perPage", per_page))

    all_locations = []
    page = 1
    while True:
        page_params = base_params + [("page", page)]
        data = cqc_get("/locations", params=page_params)
        all_locations.extend(data["locations"])
        print(f"Page {data['page']}/{data['totalPages']} -- {len(data['locations'])} rows "
              f"(running total {len(all_locations)})")
        if not data.get("nextPageUri"):
            break
        page += 1
        time.sleep(polite_delay)

    return all_locations


kent_locations_summary = fetch_all_locations(KENT_LOCAL_AUTHORITIES)
print(f"\nTotal Kent personal-care locations (summary level): {len(kent_locations_summary)}")

In [0]:
def fetch_location_detail(location_id):
    return cqc_get(f"/locations/{location_id}")


if kent_locations_summary:
    sample_detail = fetch_location_detail(kent_locations_summary[0]["locationId"])
    print(json.dumps(sample_detail, indent=2)[:3000])
else:
    print("No locations returned above -- check the filters or the key before continuing.")